# Week 3 Day 1: AFL Data Foundations
## EDA, Feature Engineering & Prediction Targets

**Tasks:**
1. Data Inventory & Understanding
2. Define Prediction Targets
3. Exploratory Data Analysis
4. Feature Engineering
5. Train/Hold-Out Split

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

data_path = r"C:\Internship\Netixsol\week-3\day-1\data\afl_datasets"

players_info = pd.read_csv(os.path.join(data_path, "afl_players_info_raw.csv"))
round_stats = pd.read_csv(os.path.join(data_path, "afl_players_round_by_round_stats_raw - afl_players_round_by_round_stats_raw.csv.csv"))
seasonal_stats = pd.read_csv(os.path.join(data_path, "afl_players_seasonal_stats_raw.csv"), low_memory=False)
matches = pd.read_csv(os.path.join(data_path, "team_matches_home_away_raw - team_matches_home_away_raw.csv.csv"))

print("Data loaded successfully!")
print(f"Players Info: {players_info.shape}")
print(f"Round Stats: {round_stats.shape}")
print(f"Seasonal Stats: {seasonal_stats.shape}")
print(f"Matches: {matches.shape}")

In [ ]:
# Data Quality Checks
for name, df in [("Players Info", players_info), ("Round Stats", round_stats), ("Matches", matches)]:
    missing = df.isnull().sum().sum()
    dupes = df.duplicated().sum()
    print(f"{name}: {missing} missing, {dupes} duplicates")

# Fix issues
for df in [players_info, round_stats, matches]:
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].str.strip()

players_info = players_info.drop_duplicates()
round_stats = round_stats.drop_duplicates()
matches = matches.drop_duplicates()
round_stats.loc[round_stats['disposals'] < 0, 'disposals'] = np.nan

print("\nData cleaned!")

In [ ]:
# Create targets
home_matches = matches[matches['home_away'] == 'H'].copy()
home_matches['home_win'] = (home_matches['result'] == 'W').astype(int)
home_matches['margin'] = home_matches['team_score'] - home_matches['opponent_score']

print(f"Home win rate: {home_matches['home_win'].mean():.1%}")
print(f"Average margin: {home_matches['margin'].mean():.1f} points")

In [ ]:
# EDA: Win rates over time
team_yearly = home_matches.groupby(['team_name', 'year']).agg(
    games=('home_win', 'count'),
    wins=('home_win', 'sum')
).reset_index()
team_yearly['win_rate'] = team_yearly['wins'] / team_yearly['games']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top teams win rate
top_teams = home_matches.groupby('team_name')['home_win'].sum().nlargest(5).index
for team in top_teams:
    team_data = team_yearly[team_yearly['team_name'] == team]
    axes[0].plot(team_data['year'], team_data['win_rate'], marker='o', label=team)
axes[0].set_title('Top 5 Teams - Win Rate Over Time')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Win Rate')
axes[0].legend()

# Home advantage
team_home = home_matches.groupby('team_name')['home_win'].mean().sort_values(ascending=True)
team_home.plot(kind='barh', ax=axes[1], color='green')
axes[1].set_title('Home Win Rate by Team')
axes[1].set_xlabel('Win Rate')

plt.tight_layout()
plt.savefig('day1_team_win_rates.png', dpi=150)
plt.show()

In [ ]:
# Feature Engineering
def calculate_team_features(group):
    group = group.sort_values('match_date')
    group['win_streak'] = 0
    streak = 0
    for i in range(len(group)):
        if group.iloc[i]['result'] == 'W':
            streak += 1
        else:
            streak = 0
        group.iloc[i, group.columns.get_loc('win_streak')] = streak
    
    group['avg_score_3'] = group['team_score'].rolling(window=3, min_periods=1).mean().shift(1)
    group['avg_conceded_3'] = group['opponent_score'].rolling(window=3, min_periods=1).mean().shift(1)
    group['avg_margin_3'] = group['margin'].rolling(window=3, min_periods=1).mean().shift(1)
    group['form_5'] = group['home_win'].rolling(window=5, min_periods=1).mean().shift(1)
    group['days_rest'] = group['match_date'].diff().dt.days
    
    return group

home_matches['match_date'] = pd.to_datetime(home_matches['match_date'])
home_matches = home_matches.groupby('team_name', group_keys=False).apply(calculate_team_features)

print("Features engineered!")
print(home_matches[['team_name', 'win_streak', 'form_5', 'avg_score_3']].head())

In [ ]:
# Train/Test Split
def time_based_split(df, train_end, val_end):
    train = df[df['year'] <= train_end].copy()
    val = df[(df['year'] > train_end) & (df['year'] <= val_end)].copy()
    test = df[df['year'] > val_end].copy()
    return train, val, test

train, val, test = time_based_split(home_matches, 2022, 2023)

print(f"Train: {len(train)} matches ({train['year'].min()}-{train['year'].max()})")
print(f"Val: {len(val)} matches ({val['year'].min()}-{val['year'].max()})")
print(f"Test: {len(test)} matches ({test['year'].min()}-{test['year'].max()})")

# Save
train.to_csv('train.csv', index=False)
val.to_csv('val.csv', index=False)
test.to_csv('test.csv', index=False)
home_matches.to_csv('feature_table.csv', index=False)

print("\nData saved!")

In [ ]:
# Data Dictionary
dictionary = """
| Feature | Description | Type |
|---------|-------------|------|
| home_win | Target: 1 if home team won, 0 if lost | Binary |
| margin | Score difference (positive = home win) | Continuous |
| win_streak | Consecutive wins | Integer |
| avg_score_3 | Avg score last 3 games | Continuous |
| avg_conceded_3 | Avg conceded last 3 games | Continuous |
| form_5 | Win rate last 5 games | Continuous |
| days_rest | Days since last match | Integer |
| h2h_win_rate | Historical win rate vs opponent | Continuous |
"""
print(dictionary)